# Mask erosion loss analysis
Reproducible notebook to generate reports in `analysis/reports`:
- `mask_erosion_loss_report.csv`
- `mask_erosion_boundary_white_report.csv`
- `mask_erosion_loss_flags_kernel10.csv`

## Metric definitions
- `kernel=0` means **no erosion** (baseline mask).
- `boundary` is the 1-pixel ring of the mask. We compute it as: `inner = erode(mask, 3x3, 1)` and `boundary = mask - inner`.
- `boundary_pixels` = number of pixels in that ring (`sum(boundary)`).
- `white_pixels` = image pixels where all R,G,B channels are >= `white_threshold`.
- `white_boundary_pixels` = white pixels that also belong to `boundary`.
- `white_boundary_pct` = `100 * white_boundary_pixels / boundary_pixels`.
- `white_boundary_pct_of_image` = `100 * white_boundary_pixels / image_pixels`.
- `image_pixels` comes from image size: `height * width`.

Important interpretation:
- `white_boundary_pct` is **not** percent of the full image.
- It is only percent of the mask boundary ring.

In [5]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# paths are relative to the analysis folder
masks_dir = Path('..') / 'data' / 'PANCREAS_PREPROCESSED'
reports_dir = Path('reports')
reports_dir.mkdir(parents=True, exist_ok=True)

loss_out = reports_dir / 'mask_erosion_loss_report.csv'
boundary_out = reports_dir / 'mask_erosion_boundary_white_report.csv'

kernel_sizes = [5, 9, 10]
white_threshold = 245

mask_paths = sorted(masks_dir.glob('*_mask.png'))
print('masks_dir:', masks_dir.resolve())
print('reports_dir:', reports_dir.resolve())
print('mask files found:', len(mask_paths))

masks_dir: /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED
reports_dir: /home/daniduhnev/projects/master-thesis/analysis/reports
mask files found: 134


In [6]:
loss_rows = []

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')

    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_gray is None:
        continue

    mask_bin = (mask_gray > 0).astype(np.uint8)
    original_pixels = int(mask_bin.sum())
    if original_pixels == 0:
        continue

    for k in kernel_sizes:
        kernel = np.ones((k, k), np.uint8)
        eroded_mask = cv2.erode(mask_bin, kernel, iterations=1)
        eroded_pixels = int(eroded_mask.sum())
        lost_pixels = original_pixels - eroded_pixels
        lost_pct = 100.0 * lost_pixels / original_pixels

        loss_rows.append({
            'study_id': study_id,
            'kernel': k,
            'orig_pixels': original_pixels,
            'eroded_pixels': eroded_pixels,
            'lost_pixels': lost_pixels,
            'lost_pct': lost_pct,
        })

loss_df = pd.DataFrame(loss_rows).sort_values(['study_id', 'kernel'])
loss_df.to_csv(loss_out, index=False)

print('loss report rows:', len(loss_df))
print('saved:', loss_out.resolve())

loss report rows: 402
saved: /home/daniduhnev/projects/master-thesis/analysis/reports/mask_erosion_loss_report.csv


In [7]:
boundary_rows = []
edge_kernel = np.ones((3, 3), np.uint8)

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')
    image_path = masks_dir / f'{study_id}_image.png'

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image_bgr is None or mask_gray is None:
        continue

    mask_bin = (mask_gray > 0).astype(np.uint8)

    image_h, image_w = image_bgr.shape[:2]
    image_pixels = int(image_h * image_w)

    for k in [0] + kernel_sizes:
        if k == 0:
            current_mask = mask_bin.copy()
        else:
            kernel = np.ones((k, k), np.uint8)
            current_mask = cv2.erode(mask_bin, kernel, iterations=1)

        inner = cv2.erode(current_mask, edge_kernel, iterations=1)
        boundary = (current_mask - inner).astype(np.uint8)

        white_pixels = (
            (image_bgr[:, :, 0] >= white_threshold)
            & (image_bgr[:, :, 1] >= white_threshold)
            & (image_bgr[:, :, 2] >= white_threshold)
        )

        boundary_pixels = int(boundary.sum())
        white_boundary_pixels = int((white_pixels & (boundary == 1)).sum())
        white_image_pixels = int(white_pixels.sum())

        if boundary_pixels > 0:
            white_boundary_pct = 100.0 * white_boundary_pixels / boundary_pixels
        else:
            white_boundary_pct = 0.0

        boundary_pct_of_image = 100.0 * boundary_pixels / image_pixels
        white_boundary_pct_of_image = 100.0 * white_boundary_pixels / image_pixels
        white_image_pct = 100.0 * white_image_pixels / image_pixels

        boundary_rows.append({
            'study_id': study_id,
            'kernel': k,
            'boundary_pixels': boundary_pixels,
            'white_boundary_pixels': white_boundary_pixels,
            'white_boundary_pct': white_boundary_pct,
            'image_pixels': image_pixels,
            'boundary_pct_of_image': boundary_pct_of_image,
            'white_boundary_pct_of_image': white_boundary_pct_of_image,
            'white_image_pixels': white_image_pixels,
            'white_image_pct': white_image_pct,
        })

boundary_df = pd.DataFrame(boundary_rows).sort_values(['study_id', 'kernel'])
boundary_df.to_csv(boundary_out, index=False)

print('boundary report rows:', len(boundary_df))
print('saved:', boundary_out.resolve())

boundary report rows: 536
saved: /home/daniduhnev/projects/master-thesis/analysis/reports/mask_erosion_boundary_white_report.csv


In [12]:
print('summary')
print('')

print('loss by kernel')
print(
    loss_df.groupby('kernel')['lost_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('white boundary overlap percent by kernel (within mask boundary only)')
print(
    boundary_df.groupby('kernel')['white_boundary_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('sanity check: boundary and white-boundary as percent of full image')
print(
    boundary_df.groupby('kernel')[['boundary_pct_of_image', 'white_boundary_pct_of_image', 'white_image_pct']]
    .agg(['mean', 'median', 'max'])
    .round(4)
)

print('')
print('top 5 highest loss cases at kernel 9')
print(
    loss_df[loss_df['kernel'] == 9]
    .sort_values('lost_pct', ascending=False)
    .head(5)[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']]
)

print('')
print('top 5 highest white-boundary overlap at kernel 9')
print(
    boundary_df[boundary_df['kernel'] == 9]
    .sort_values('white_boundary_pct', ascending=False)
    .head(5)[['study_id', 'white_boundary_pixels', 'boundary_pixels', 'white_boundary_pct', 'white_boundary_pct_of_image']]
)

summary

loss by kernel
          mean  median     min     max
kernel                                
5        9.954   9.330   4.982  26.124
9       19.550  18.334   9.871  50.316
10      21.893  20.510  11.081  56.074

white boundary overlap percent by kernel (within mask boundary only)
          mean  median     min     max
kernel                                
0       49.051  49.719  27.495  81.503
5       25.563  26.331   8.780  37.500
9        0.433   0.000   0.000   2.889
10       0.398   0.000   0.000   2.715

sanity check: boundary and white-boundary as percent of full image
       boundary_pct_of_image                 white_boundary_pct_of_image  \
                        mean  median     max                        mean   
kernel                                                                     
0                     0.1056  0.1035  0.2205                      0.0510   
5                     0.1025  0.0999  0.2176                      0.0265   
9                     0.0993 

In [8]:
# flag studies with high mask loss at kernel 10
threshold_pct = 30.0

k10 = loss_df[loss_df['kernel'] == 10].copy()
flagged = k10[k10['lost_pct'] > threshold_pct].sort_values('lost_pct', ascending=False)

print('kernel 10 studies:', len(k10))
print('threshold (%):', threshold_pct)
print('flagged studies:', len(flagged))
print('flagged percent:', round(100.0 * len(flagged) / len(k10), 2), '%')

print('')
print('kernel 10 lost_pct quantiles')
print(k10['lost_pct'].quantile([0.75, 0.80, 0.85, 0.90, 0.95]).round(3))

print('')
print('flagged studies list')
print(flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']])

flag_out = reports_dir / 'mask_erosion_loss_flags_kernel10.csv'
flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']].to_csv(flag_out, index=False)
print('')
print('saved:', flag_out.resolve())

kernel 10 studies: 134
threshold (%): 30.0
flagged studies: 15
flagged percent: 11.19 %

kernel 10 lost_pct quantiles
0.75    25.000
0.80    26.264
0.85    27.641
0.90    31.353
0.95    36.460
Name: lost_pct, dtype: float64

flagged studies list
    study_id  orig_pixels  lost_pixels   lost_pct
227    28_03         2848         1597  56.074438
215    27_01         5964         2697  45.221328
116    14_01         5511         2362  42.859735
260    33_03         3585         1456  40.613668
401    56_03         8421         3301  39.199620
140    16_01         5040         1903  37.757937
62     08_01        10202         3739  36.649677
56     07_02         8078         2937  36.358009
14     01_05         7983         2839  35.563072
389    55_02         3811         1282  33.639465
47     06_03        10325         3430  33.220339
50     06_04         9279         2942  31.706003
20     02_02         6108         1932  31.630648
269    35_01         5828         1834  31.468771
59  

In [13]:
# Quick verification of where key numbers come from

# 1) Image size used in this dataset
print('image_pixels unique values:', boundary_df['image_pixels'].nunique())
print('image_pixels value:', int(boundary_df['image_pixels'].iloc[0]))

# 2) Kernel 0 averages (same logic discussed above)
k0 = boundary_df[boundary_df['kernel'] == 0]
mean_boundary_pixels = k0['boundary_pixels'].mean()
mean_white_boundary_pixels = k0['white_boundary_pixels'].mean()
mean_white_boundary_pct = k0['white_boundary_pct'].mean()
mean_white_boundary_pct_of_image = k0['white_boundary_pct_of_image'].mean()

print('')
print('kernel 0 mean boundary_pixels:', round(mean_boundary_pixels, 3))
print('kernel 0 mean white_boundary_pixels:', round(mean_white_boundary_pixels, 3))
print('kernel 0 mean white_boundary_pct (within boundary):', round(mean_white_boundary_pct, 3))
print('kernel 0 mean white_boundary_pct_of_image:', round(mean_white_boundary_pct_of_image, 4), '%')

# 3) Show one concrete study calculation
example = k0.iloc[0]
print('')
print('example study:', example['study_id'])
print('boundary_pixels:', int(example['boundary_pixels']))
print('white_boundary_pixels:', int(example['white_boundary_pixels']))
print(
    'white_boundary_pct = 100 * white_boundary_pixels / boundary_pixels =',
    round(100.0 * int(example['white_boundary_pixels']) / int(example['boundary_pixels']), 6)
)
print(
    'white_boundary_pct_of_image = 100 * white_boundary_pixels / image_pixels =',
    round(100.0 * int(example['white_boundary_pixels']) / int(example['image_pixels']), 6)
)

image_pixels unique values: 1
image_pixels value: 786432

kernel 0 mean boundary_pixels: 830.396
kernel 0 mean white_boundary_pixels: 400.791
kernel 0 mean white_boundary_pct (within boundary): 49.051
kernel 0 mean white_boundary_pct_of_image: 0.051 %

example study: 01_01
boundary_pixels: 1030
white_boundary_pixels: 586
white_boundary_pct = 100 * white_boundary_pixels / boundary_pixels = 56.893204
white_boundary_pct_of_image = 100 * white_boundary_pixels / image_pixels = 0.074514
